In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/processed/supply_chain_cleaned.csv",
    encoding="latin1"
)

print("Dataset Shape:", df.shape)

Dataset Shape: (180519, 51)


In [2]:
# Exclude cancelled orders

df_model = df[
    df["Delivery Status"] != "Shipping canceled"
].copy()

print("Original Records:", len(df))
print("Records after excluding cancelled:", len(df_model))

Original Records: 180519
Records after excluding cancelled: 172765


In [3]:
# Create target variable

df_model["Target"] = np.where(
    df_model["Delivery Status"] == "Late delivery",
    1,
    0
)

print("Target Distribution:")
print(df_model["Target"].value_counts())

print("\nTarget Percentage:")
print(
    (df_model["Target"].value_counts(normalize=True) * 100).round(2)
)

Target Distribution:
Target
1    98977
0    73788
Name: count, dtype: int64

Target Percentage:
Target
1    57.29
0    42.71
Name: proportion, dtype: float64


In [5]:
# step 2.1 Convert Order Date into datetime

df_model["order date (DateOrders)"] = pd.to_datetime(
    df_model["order date (DateOrders)"],
    errors="coerce",
    format="mixed"
)

print(df_model["order date (DateOrders)"].dtype)

print(
    "Invalid Dates:",
    df_model["order date (DateOrders)"].isnull().sum()
)

datetime64[us]
Invalid Dates: 0


In [6]:
# step 2.2  create new data feature

order_date = df_model["order date (DateOrders)"]

df_model["Order_Year"] = order_date.dt.year
df_model["Order_Month"] = order_date.dt.month
df_model["Order_Day"] = order_date.dt.day
df_model["Order_Weekday"] = order_date.dt.dayofweek

df_model["Is_Weekend"] = (
    order_date.dt.dayofweek >= 5
).astype("int")

print(
    df_model[
        [
            "order date (DateOrders)",
            "Order_Year",
            "Order_Month",
            "Order_Day",
            "Order_Weekday",
            "Is_Weekend"
        ]
    ].head(10)
)

   order date (DateOrders)  Order_Year  Order_Month  Order_Day  Order_Weekday  \
0      2018-01-31 22:56:00        2018            1         31              2   
1      2018-01-13 12:27:00        2018            1         13              5   
2      2018-01-13 12:06:00        2018            1         13              5   
3      2018-01-13 11:45:00        2018            1         13              5   
4      2018-01-13 11:24:00        2018            1         13              5   
6      2018-01-13 10:42:00        2018            1         13              5   
7      2018-01-13 10:21:00        2018            1         13              5   
8      2018-01-13 10:00:00        2018            1         13              5   
9      2018-01-13 09:39:00        2018            1         13              5   
11     2018-01-13 08:57:00        2018            1         13              5   

    Is_Weekend  
0            0  
1            1  
2            1  
3            1  
4            1  
6     

In [7]:
# step 2.3 verify

print("New Dataset Shape:", df_model.shape)

print("\nNew Features:")
print(df_model[
    [
        "Order_Year",
        "Order_Month",
        "Order_Day",
        "Order_Weekday",
        "Is_Weekend"
    ]
].describe())


New Dataset Shape: (172765, 57)

New Features:
          Order_Year    Order_Month      Order_Day  Order_Weekday  \
count  172765.000000  172765.000000  172765.000000  172765.000000   
mean     2015.970874       6.235511      15.739803       3.009209   
std         0.829255       3.405593       8.823467       2.000183   
min      2015.000000       1.000000       1.000000       0.000000   
25%      2015.000000       3.000000       8.000000       1.000000   
50%      2016.000000       6.000000      16.000000       3.000000   
75%      2017.000000       9.000000      23.000000       5.000000   
max      2018.000000      12.000000      31.000000       6.000000   

          Is_Weekend  
count  172765.000000  
mean        0.287020  
std         0.452372  
min         0.000000  
25%         0.000000  
50%         0.000000  
75%         1.000000  
max         1.000000  


In [8]:
# step 3.1 identified categorical columns

categorical_columns = df_model.select_dtypes(
    include=["object", "str", "category"]
).columns

print("Total Categorical Columns:", len(categorical_columns))

for col in categorical_columns:
    print(
        f"{col}: {df_model[col].nunique()} unique values"
    )

Total Categorical Columns: 23
Type: 4 unique values
Delivery Status: 3 unique values
Category Name: 50 unique values
Customer City: 563 unique values
Customer Country: 2 unique values
Customer Email: 1 unique values
Customer Fname: 781 unique values
Customer Lname: 1109 unique values
Customer Password: 1 unique values
Customer Segment: 3 unique values
Customer State: 46 unique values
Customer Street: 7435 unique values
Department Name: 11 unique values
Market: 5 unique values
Order City: 3585 unique values
Order Country: 164 unique values
Order Region: 23 unique values
Order State: 1083 unique values
Order Status: 7 unique values
Product Image: 118 unique values
Product Name: 118 unique values
shipping date (DateOrders): 61024 unique values
Shipping Mode: 4 unique values


In [9]:
# step 3.2 important categorical columns inspecting

selected_columns = [
    "Shipping Mode",
    "Customer Segment",
    "Market",
    "Order Region",
    "Type",
    "Department Name"
]

for col in selected_columns:
    print(f"\n{col}")
    print(df_model[col].value_counts().head(10))


Shipping Mode
Shipping Mode
Standard Class    103153
Second Class       33806
First Class        26513
Same Day            9293
Name: count, dtype: int64

Customer Segment
Customer Segment
Consumer       89420
Corporate      52528
Home Office    30817
Name: count, dtype: int64

Market
Market
LATAM           49309
Europe          48090
Pacific Asia    39585
USCA            24627
Africa          11154
Name: count, dtype: int64

Order Region
Order Region
Central America    27174
Western Europe     25867
South America      14184
Oceania             9733
Northern Europe     9408
Southeast Asia      9136
Southern Europe     9030
Caribbean           7951
West of USA         7595
South Asia          7455
Name: count, dtype: int64

Type
Type
DEBIT       69295
TRANSFER    42129
PAYMENT     41725
CASH        19616
Name: count, dtype: int64

Department Name
Department Name
Fan Shop      64033
Apparel       46884
Golf          31768
Footwear      13891
Outdoors       9267
Fitness        2374
Discs

In [10]:
# step 4.1 feature selection

selected_features = [
    "Shipping Mode",
    "Days for shipment (scheduled)",
    "Customer Segment",
    "Market",
    "Order Region",
    "Department Name",
    "Category Name",
    "Order Item Quantity",
    "Order Item Discount",
    "Order Item Discount Rate",
    "Order Item Product Price",
    "Order_Month",
    "Order_Weekday",
    "Is_Weekend"
]

X = df_model[selected_features].copy()

y = df_model["Target"].copy()

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nSelected Features:")
print(X.columns.tolist())

Features Shape: (172765, 14)
Target Shape: (172765,)

Selected Features:
['Shipping Mode', 'Days for shipment (scheduled)', 'Customer Segment', 'Market', 'Order Region', 'Department Name', 'Category Name', 'Order Item Quantity', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order_Month', 'Order_Weekday', 'Is_Weekend']


In [11]:
print("Feature Data Types:")
print(X.dtypes)

print("\nTarget Distribution:")
print(y.value_counts())

Feature Data Types:
Shipping Mode                        str
Days for shipment (scheduled)      int64
Customer Segment                     str
Market                               str
Order Region                         str
Department Name                      str
Category Name                        str
Order Item Quantity                int64
Order Item Discount              float64
Order Item Discount Rate         float64
Order Item Product Price         float64
Order_Month                        int32
Order_Weekday                      int32
Is_Weekend                         int64
dtype: object

Target Distribution:
Target
1    98977
0    73788
Name: count, dtype: int64


In [12]:
# Step 5: Separate numerical and categorical features

categorical_features = X.select_dtypes(
    include=["object", "str", "category", "string"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

Categorical Features:
['Shipping Mode', 'Customer Segment', 'Market', 'Order Region', 'Department Name', 'Category Name']

Numerical Features:
['Days for shipment (scheduled)', 'Order Item Quantity', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order_Month', 'Order_Weekday', 'Is_Weekend']


In [13]:
# Train-Test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

Training Features: (138212, 14)
Testing Features: (34553, 14)
Training Target: (138212,)
Testing Target: (34553,)


In [14]:
# Encoding and scaling pipeline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both transformers
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing Pipeline Created Successfully!")

Preprocessing Pipeline Created Successfully!


In [15]:
# Transform training and testing data

# Fit only on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Apply same transformations to testing data
X_test_processed = preprocessor.transform(X_test)

print("Training Data Shape:", X_train_processed.shape)
print("Testing Data Shape:", X_test_processed.shape)

Training Data Shape: (138212, 104)
Testing Data Shape: (34553, 104)


In [16]:
import joblib
import os

# Models folder ensure karo
os.makedirs("../models", exist_ok=True)

# Preprocessor save karo
joblib.dump(preprocessor, "../models/preprocessor.pkl")

print("Preprocessor saved successfully!")

Preprocessor saved successfully!


In [17]:
# verify

import os

print("File exists:", os.path.exists("../models/preprocessor.pkl"))
print("Training data shape:", X_train_processed.shape)
print("Testing data shape:", X_test_processed.shape)
print("Target training shape:", y_train.shape)
print("Target testing shape:", y_test.shape)

File exists: True
Training data shape: (138212, 104)
Testing data shape: (34553, 104)
Target training shape: (138212,)
Target testing shape: (34553,)
